# Semantic Search using PineCone DB (Cloud Based)

website: https://www.pinecone.io/

## Objective

1) Convert text → embeddings

2) Store in vector database

3) Perform semantic search

In [ ]:
# I used following on  command line: Took 15 minutes
python -m pip install pinecone==10.0.0

In [1]:
import pinecone

# Print the installed package version
print("Pinecone Library Version:", pinecone.__version__)

Pinecone Library Version: 10.0.0


In [ ]:
# Import Librarairs: Took a 4-5 minutes

from pinecone import Pinecone, ServerlessSpec


In [3]:
from importlib import metadata
from pinecone import Pinecone, ServerlessSpec

# Get version via package metadata
version = metadata.version("pinecone")
print(version) # 7.3.0

10.0.0


In [4]:
import os
from getpass import getpass

# 1. Prompt the user for their key securely (input will be hidden as they type)
PINECONE_API_KEY = getpass("Enter your Pinecone API Key: ")

# 2. Store it as an environment variable so your Pinecone client can read it
# os.environ["PINECONE_API_KEY"] = pinecone_key

print("PINECONE_API_KEY has been successfully set in the environment!")


Enter your Pinecone API Key:  ········


PINECONE_API_KEY has been successfully set in the environment!


In [5]:
# Need API Key
# PINECONE_API_KEY = "pcsk_key_here"
pc = Pinecone(api_key=PINECONE_API_KEY)
# pc = Pinecone(api_key="pcsk_KEY")

print("🔄 Testing Pinecone API Key connectivity...")

try:
    # 2. Trigger a control plane API call to verify the key
    active_indexes = pc.list_indexes()
    
    print("✅ Success! Your Pinecone API key is valid and authenticated.")
    print(f"📁 Available Indexes in your project: {len(active_indexes)}")
    for index in active_indexes:
        print(f"   - {index.name} ({index.dimension}d, {index.metric})")

except PineconeException as e:
    # 3. Handle specific authentication/network errors safely
    print("❌ Authentication Failed!")
    print(f"🔍 Error details: {str(e)}")

🔄 Testing Pinecone API Key connectivity...
✅ Success! Your Pinecone API key is valid and authenticated.
📁 Available Indexes in your project: 4
   - company-policies-index (1536d, cosine)
   - product-manuals-index (1536d, cosine)
   - loxford-quantum-mechanics (1536d, euclidean)
   - rag-chatbot-industry (1536d, cosine)


# Create an index

In [6]:
index_name = "genai-demo"

if not pc.has_index(index_name):
    pc.create_index_for_model(
        name=index_name,
        cloud="aws",
        region="us-east-1",
        embed={
            "model":"llama-text-embed-v2",
            "field_map":{"text": "chunk_text"}
        }
    )

# Connect to Index

In [7]:
index = pc.Index(index_name)

# Prepare the documents

In [8]:
# Contains 2 main topics: AI and Cars
documents = [
    "Machine learning is a subset of AI. It is easy to learn",
    "Deep learning uses neural networks",
    "Python is used for data science",
    "Cars and vehicles are transportation",
    "Artificial intelligence is transforming industries",
    "Football is a popular sport",
    "Data engineering involves pipelines",
    "Rides are good mode of transportation",
    "Cars and automobiles are expensive",
    "Healthcare is improved by AI diagnostics. It is bringing positive change in the world",
    "Self-driving vehicles are te future"
]

In [11]:
chunks = []
for i, doc in enumerate(documents):
    chunk_item = {
        "_id": str(i),
        "chunk_text": doc
    }
    chunks.append(chunk_item)

In [12]:
chunks

[{'_id': '0',
  'chunk_text': 'Machine learning is a subset of AI. It is easy to learn'},
 {'_id': '1', 'chunk_text': 'Deep learning uses neural networks'},
 {'_id': '2', 'chunk_text': 'Python is used for data science'},
 {'_id': '3', 'chunk_text': 'Cars and vehicles are transportation'},
 {'_id': '4',
  'chunk_text': 'Artificial intelligence is transforming industries'},
 {'_id': '5', 'chunk_text': 'Football is a popular sport'},
 {'_id': '6', 'chunk_text': 'Data engineering involves pipelines'},
 {'_id': '7', 'chunk_text': 'Rides are good mode of transportation'},
 {'_id': '8', 'chunk_text': 'Cars and automobiles are expensive'},
 {'_id': '9',
  'chunk_text': 'Healthcare is improved by AI diagnostics. It is bringing positive change in the world'},
 {'_id': '10', 'chunk_text': 'Self-driving vehicles are te future'}]

# Store the documents in vector DB

In [13]:
# Upsert the records into a namespace

index.upsert_records(
    namespace="example-namespace", 
    records=chunks
)

UpsertRecordsResponse(record_count=11, response_info=ResponseInfo(raw_headers={'date': 'Sat, 05 Sep 2026 14:18:24 GMT', 'content-length': '0', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '1', 'x-pinecone-api-version': '2026-07', 'x-pinecone-request-latency-ms': '594', 'x-envoy-upstream-service-time': '594', 'x-pinecone-response-duration-ms': '595', 'server': 'envoy'}))

In [14]:
# Wait for the upserted vectors to be indexed
import time
time.sleep(10)

# View stats for the index
stats_for_index = index.describe_index_stats()
print(stats_for_index)

DescribeIndexStatsResponse(dimension=1024, total_vector_count=11, metric='cosine', namespaces=1)


# Perform Semantic Search

In [15]:
# Define the query
query = "how is AI is changing the world ?"

# Search the dense index
results = index.search(
    namespace="example-namespace",
    query={
        "top_k": 10,
        "inputs": {
            'text': query
        }
    }
)

In [16]:
# Returns results sorted by similarity score
results

SearchRecordsResponse(result=SearchResult(hits=[Hit(id='9', score=0.4743359684944153, fields={'chunk_text': 'Healthcare is improved by AI diagnostics. It is bringing positive change in the world'}), Hit(id='4', score=0.31615668535232544, fields={'chunk_text': 'Artificial intelligence is transforming industries'}), Hit(id='0', score=0.164500892162323, fields={'chunk_text': 'Machine learning is a subset of AI. It is easy to learn'}), Hit(id='10', score=0.12512488663196564, fields={'chunk_text': 'Self-driving vehicles are te future'}), Hit(id='1', score=0.10879135876893997, fields={'chunk_text': 'Deep learning uses neural networks'}), Hit(id='2', score=0.09779620915651321, fields={'chunk_text': 'Python is used for data science'}), Hit(id='6', score=0.0435304269194603, fields={'chunk_text': 'Data engineering involves pipelines'}), Hit(id='3', score=0.02478351816534996, fields={'chunk_text': 'Cars and vehicles are transportation'}), Hit(id='8', score=-0.004705571103841066, fields={'chunk_te

In [19]:
# Print the results

for hit in results['result']['hits']: # returns result sorted by score
    # print(hit)
    # print(f"id: {hit['_id']:<5} | score: {round(hit['_score'], 2):<5} | text: {hit['fields']['chunk_text']:<50}")
    print(f"id: {hit['id']:<5} | score: {round(hit['score'], 2):<5} | text: {hit['fields']['chunk_text']:<50}")


id: 9     | score: 0.47  | text: Healthcare is improved by AI diagnostics. It is bringing positive change in the world
id: 4     | score: 0.32  | text: Artificial intelligence is transforming industries
id: 0     | score: 0.16  | text: Machine learning is a subset of AI. It is easy to learn
id: 10    | score: 0.13  | text: Self-driving vehicles are te future               
id: 1     | score: 0.11  | text: Deep learning uses neural networks                
id: 2     | score: 0.1   | text: Python is used for data science                   
id: 6     | score: 0.04  | text: Data engineering involves pipelines               
id: 3     | score: 0.02  | text: Cars and vehicles are transportation              
id: 8     | score: -0.0  | text: Cars and automobiles are expensive                
id: 5     | score: -0.01 | text: Football is a popular sport                       


In [21]:
# Lets use a different query: on cars
query = "Tell me about rides ?"

# Search the dense index
results = index.search(
    namespace="example-namespace",
    query={
        "top_k": 10,
        "inputs": {
            'text': query
        }
    }
)

# Print the results
for hit in results['result']['hits']: # returns result sorted by score
    # print(hit)
    # print(f"id: {hit['_id']:<5} | score: {round(hit['_score'], 2):<5} | text: {hit['fields']['chunk_text']:<50}")
    print(f"id: {hit['id']:<5} | score: {round(hit['score'], 2):<5} | text: {hit['fields']['chunk_text']:<50}")


id: 7     | score: 0.26  | text: Rides are good mode of transportation             
id: 3     | score: 0.08  | text: Cars and vehicles are transportation              
id: 8     | score: 0.05  | text: Cars and automobiles are expensive                
id: 10    | score: 0.05  | text: Self-driving vehicles are te future               
id: 5     | score: 0.04  | text: Football is a popular sport                       
id: 9     | score: -0.01 | text: Healthcare is improved by AI diagnostics. It is bringing positive change in the world
id: 0     | score: -0.01 | text: Machine learning is a subset of AI. It is easy to learn
id: 4     | score: -0.02 | text: Artificial intelligence is transforming industries
id: 6     | score: -0.03 | text: Data engineering involves pipelines               
id: 1     | score: -0.04 | text: Deep learning uses neural networks                


# Delete the index when done

In [22]:
pc.delete_index(index_name)